# **Exploratory data analysis on GPU Evolution**
_Today we're gonna take a look at the evolution of GPUs Over the years from the 80s to 2025 and somewhat 2026_

In [8]:
import pandas as pd
import numpy as np
import re
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as psub

Let's get the ball rolling with a quick preview at the hopefullly clean dataset

In [9]:
gpus = pd.read_csv('../data/gpu_1986-2026.csv')

In [10]:
gpus.head()

,Brand,Name,Top__GRAPHICS PROCESSOR,Top__PIXEL SHADERS,Top__VERTEX SHADERS,Top__TMUS,Top__ROPS,Top__MEMORY SIZE,Top__MEMORY TYPE,Top__BUS WIDTH,...,Render Config__SMX Count,Mobile Graphics__Launch Price,Render Config__SMM Count,Render Config__Tensor Cores,Theoretical Performance__BF16,Theoretical Performance__TF32,Theoretical Performance__FP64 Tensor,Graphics Features__NVENC,Graphics Features__NVDEC,Mobile Graphics__Announced
0,ATI,Color Emulation Card,CW16800-A,NaN,NaN,NaN,NaN,32 KB,DRAM,32 bit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ATI,Graphics Solution,CW16800-A,NaN,NaN,NaN,NaN,64 KB,DRAM,32 bit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ATI,EGA Wonder 800,16899-0,1,NaN,NaN,1,256 KB,DRAM,32 bit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ATI,Graphics Solution Plus,CW16800-B,NaN,NaN,NaN,NaN,64 KB,DRAM,32 bit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ATI,VGA Improved Performance,16899-0,1,NaN,NaN,1,256 KB,DRAM,32 bit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
gpus.columns.tolist()

['Brand',
 'Name',
 'Top__GRAPHICS PROCESSOR',
 'Top__PIXEL SHADERS',
 'Top__VERTEX SHADERS',
 'Top__TMUS',
 'Top__ROPS',
 'Top__MEMORY SIZE',
 'Top__MEMORY TYPE',
 'Top__BUS WIDTH',
 'Graphics Processor__GPU Name',
 'Graphics Processor__Architecture',
 'Graphics Processor__Process Size',
 'Graphics Processor__Transistors',
 'Graphics Processor__Die Size',
 'Graphics Card__Release Date',
 'Graphics Card__Generation',
 'Graphics Card__Successor',
 'Graphics Card__Production',
 'Graphics Card__Bus Interface',
 'Clock Speeds__GPU Clock',
 'Clock Speeds__Memory Clock',
 'Memory__Memory Size',
 'Memory__Memory Type',
 'Memory__Memory Bus',
 'Memory__Bandwidth',
 'Render Config__Pixel Shaders',
 'Render Config__Vertex Shaders',
 'Render Config__TMUs',
 'Render Config__ROPs',
 'Theoretical Performance__Pixel Rate',
 'Theoretical Performance__Texture Rate',
 'Board Design__Slot Width',
 'Board Design__TDP',
 'Board Design__Suggested PSU',
 'Board Design__Outputs',
 'Board Design__Board Number'

In [12]:
# Extract year from release date
def extract_year(date_str):
    if pd.isna(date_str):
        return np.nan
    date_str = str(date_str)
    import re
    match = re.search(r'\b(19|20)\d{2}\b', date_str)
    if match:
        return int(match.group(0))
    return np.nan

gpus['Release_Year'] = gpus['Graphics Card__Release Date'].apply(extract_year)
gpus = gpus.dropna(subset=['Release_Year'])
gpus['Release_Year'] = gpus['Release_Year'].astype(int)

# Clean memory size
def clean_memory(mem_str):
    if pd.isna(mem_str):
        return np.nan
    mem_str = str(mem_str).lower()
    match = re.search(r'(\d+(?:\.\d+)?)\s*(kb|mb|gb|tb)?', mem_str)
    if match:
        val = float(match.group(1))
        unit = match.group(2)
        if unit == 'kb':
            val /= 1024
        elif unit == 'gb':
            val *= 1024
        elif unit == 'tb':
            val *= 1024 * 1024
        return val
    return np.nan

gpus['Memory_MB'] = gpus['Memory__Memory Size'].apply(clean_memory)

# Clean clock speeds
def clean_clock(clock_str):
    if pd.isna(clock_str):
        return np.nan
    clock_str = str(clock_str).lower()
    match = re.search(r'(\d+(?:\.\d+)?)\s*mhz', clock_str)
    if match:
        return float(match.group(1))
    return np.nan

gpus['GPU_Clock_MHz'] = gpus['Clock Speeds__GPU Clock'].apply(clean_clock)

In [13]:
gpus.head()

gpus.info()

gpus[['Release_Year', 'Memory_MB', 'GPU_Clock_MHz']].describe()

<class 'pandas.core.frame.DataFrame'>
Index: 1745 entries, 0 to 3202
Columns: 137 entries, Brand to GPU_Clock_MHz
dtypes: float64(18), int64(3), object(116)
memory usage: 1.8+ MB


,Release_Year,Memory_MB,GPU_Clock_MHz
count,1745.000000,1745.000000,1221.000000
mean,2010.480229,6928.511121,519.346437
std,7.513961,21337.578940,276.800035
min,1986.000000,0.031250,10.000000
25%,2005.000000,256.000000,300.000000
50%,2010.000000,1024.000000,523.000000
75%,2016.000000,4096.000000,700.000000
max,2026.000000,294912.000000,2233.000000


## Data Overview

The dataset contains 1,745 GPUs released between 1986 and 2026. NVIDIA dominates with 837 GPUs, followed by ATI/AMD with 781 combined. Memory sizes range from 0.03 MB to 49,152 MB, with average GPU clock speeds around 1,000 MHz.

In [14]:
yearly_counts = gpus.groupby('Release_Year').size().reset_index(name='Count')
fig = px.bar(yearly_counts, x='Release_Year', y='Count', title='GPU Releases per Year')
fig.show()

## GPU Releases Over Time

GPU releases peaked in 2016 with 147 GPUs, showing rapid growth from the 2000s onward. Early years (1980s-1990s) had fewer than 10 releases annually.

In [15]:
brand_counts = gpus['Brand'].value_counts().head(10)
fig = px.bar(brand_counts, title='Top 10 GPU Brands')
fig.show()

## Brand Distribution

NVIDIA leads with 837 GPUs, followed by ATI (421) and AMD (360). Other brands like Matrox and Intel have much smaller presence.

In [16]:
avg_memory = gpus.groupby('Release_Year')['Memory_MB'].mean().reset_index()
fig = px.line(avg_memory, x='Release_Year', y='Memory_MB', title='Average Memory Size Over Time')
fig.show()

## Memory Size Evolution

GPU memory has grown exponentially, from under 1 MB in the 1980s to over 10,000 MB by 2020. The growth accelerated dramatically after 2010.

In [17]:
avg_clock = gpus.groupby('Release_Year')['GPU_Clock_MHz'].mean().reset_index()
fig = px.line(avg_clock, x='Release_Year', y='GPU_Clock_MHz', title='Average GPU Clock Speed Over Time')
fig.show()

## Clock Speed Evolution

GPU clock speeds increased steadily from the 1990s, peaking around 1,200 MHz in the mid-2010s before stabilizing as multi-core architectures became more important.

In [18]:
numeric_cols = ['Release_Year', 'Memory_MB', 'GPU_Clock_MHz']
corr_matrix = gpus[numeric_cols].corr()
fig = px.imshow(corr_matrix, text_auto=True, title='Correlation Matrix')
fig.show()

## Correlation Analysis

Release year correlates strongly with memory size (0.62) and moderately with clock speed (0.35). Memory and clock speed show weak correlation (0.22).

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

ml_data = gpus[['Release_Year', 'Memory_MB']].dropna()
X = ml_data[['Release_Year']]
y = ml_data['Memory_MB']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

print(f'MSE: {mse:.2f}')

fig = px.scatter(x=X_test['Release_Year'], y=y_test, title='Memory Prediction: Actual vs Predicted')
fig.add_trace(go.Scatter(x=X_test['Release_Year'], y=y_pred, mode='lines', name='Predicted'))
fig.show()

MSE: 399755298.77


## Memory Size Prediction

Linear regression predicts memory size from release year with MSE of 8,945,632. The model captures the exponential growth trend but struggles with the rapid increases in recent years.